<!-- DATA PROVIDER INSTRUCTIONS

1. Provide the name of your dataset, replacing the bracketed placeholder text.
2. Update the Registry of Open Data landing page URL, by replacing the bracketed placeholder text. The [REGISTRY_YAML_NAME] will correspond to the name of the YAML document in your pull request to the Registry of Open Data on Github, minus the .yaml file extension.
3. Remove these comment blocks when you have completed each section.

DATA PROVIDER INSTRUCTIONS -->

# Get to Know a Dataset: met-office-cmip6

This notebook serves as a guided tour of the [met-office-cmip6](https://registry.opendata.aws/met-office-cmip6) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

<!-- DATA PROVIDER INSTRUCTIONS

The goal of this section is to orient users to the structure of your dataset. 

1. How are key prefixes and objects organized in your S3 bucket?
2. What kinds of filetypes are represented in your dataset?
3. Explain with text what users are expected to encounter, and then demonstrate with code the organizational framework you applied when creating your dataset.
4. The responses to each question section are meant to be expanded or replaced as dictated by your dataset

DATA PROVIDER INSTRUCTIONS -->

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.


This data is a replica of hindcast data published for the Decadal Climate Prediction Project by the Met Office data to CMIP6 via [ESGF](https://esgf-ui.ceda.ac.uk/search).
Data is stored in collections of netCDF4 files structured and organised according to the Directory Reference Syntax recorded in the [Global Attributes documentation for CMIP6](https://zenodo.org/records/15670624).

The directory structure is illustrated below

```
CMIP6/
    CMIP/MOHC/HadGEM3-GC31-MM/piControl/
        r1i1p1f1/
            fx/
                [variable name]/
                    gn/vYYYYMMDD/
                        [files]
            Ofx/
                [variable name]/
                    gn/vYYYYMMDD/
                        [files]
    DCPP/MOHC/HadGEM3-GC31-MM/dcppA-hindcast/
        s<start year>-r<ensemble index>i1p1f2/
            [table id]/
                [variable name]/
                    gn/vYYYYMMDD/
                        [files]
```

The "fixed" variables, i.e. those that do not vary in time, included from the HadGEM3-GC31-MM piControl (pre-industrial control) simulation include [orography (fx/orog)](https://met-office-cmip6.s3-eu-west-2.amazonaws.com/CMIP6/CMIP/MOHC/HadGEM3-GC31-MM/piControl/r1i1p1f1/fx/orog/gn/v20200108/orog_fx_HadGEM3-GC31-MM_piControl_r1i1p1f1_gn.nc) and [land-sea mask](https://met-office-cmip6.s3-eu-west-2.amazonaws.com/CMIP6/CMIP/MOHC/HadGEM3-GC31-MM/piControl/r1i1p1f1/fx/sftlf/gn/v20200108/sftlf_fx_HadGEM3-GC31-MM_piControl_r1i1p1f1_gn.nc).

The hindcast data includes simulations starting each year between 1960 and 2018, with 10 ensemble members for each year. For example to obtain the monthly mean surface temperature for the seventh simulation starting in 1998 look [here](https://met-office-cmip6.s3.eu-west-2.amazonaws.com/index.html#CMIP6/DCPP/MOHC/HadGEM3-GC31-MM/dcppA-hindcast/s1961-r1i1p1f2/Amon/tas/gn/v20200417/).

Files are named using the following naming convention (see link to Global Attributes document above for full details);

```
<variable name>_<table id>_HadGEM3-GC31-MM_dcppA-hindcast_s<start-year>-r<ensemble index>i1p1f2_gn_<start date>-<end date>.nc
```
Data for each year is stored separately and the start and end date format is `YYYYMM` for most files with the exception of those in the `day` table (daily frequency) for which the format is `YYYYMMDD`.

**Important note: This model uses a 360-day calendar where each month has 30 days.  Some tools will struggle to interpret the dates correctly, but [CF units](https://cf-units.readthedocs.io/en/latest/utilities.html) can be used with all CF compatible data**

A list of variables and their properties can be found [here](https://github.com/MetOffice/ASDI-CMIP6-DCPP-hindcast/blob/main/variable_descriptions.csv).

Each file in this dataset has been "repacked" using [cmip7repack](https://github.com/NCAS-CMS/cmip7_repack) to optimise read performance particularly within cloud environments.


First we will import the Python libraries required throughout this notebook.



In [ ]:
# This notebook was constructed using the following additional libraries. Other versions may work but have not been tested
# (please install using the preferred method for your environment, e.g. pip, conda):
# s3fs = 2026.6.0
# fsspec = 2026.6.0
# xarray = 2026.7.0
# matplotlib = 3.11.1
# boto3 = 1.43.46

# Import the libraries required for this notebook
# Built-ins

# Installed libraries
import s3fs
import fsspec
import xarray as xr
from matplotlib import pyplot as plt

import boto3 
from botocore import UNSIGNED
from botocore.config import Config



Next, we will define the location of our dataset, create our boto3 S3 client, and list the top level prefixes in our S3 bucket. Here we see there is only one top-level prefix in our bucket.

In [ ]:
# Location of the S3 bucket for this dataset
bucket = "met-office-cmip6"

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests.
# Here we set the signature version to unsigned, which is required for public buckets.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Print the items in the top-level prefixes
for item in s3.list_objects_v2(Bucket=bucket, Delimiter='/')['CommonPrefixes']:
    print(item['Prefix'])


Fixed field information, such as orography altitude (orog), fractional land sea masks (sftlf) and grid cell area (areacella/o) can be found under the location for the piControl (pre-industrial control) simulation:

In [ ]:
print('Atmosphere Fixed field locations:')

for item in s3.list_objects_v2(Bucket=bucket, Prefix='CMIP6/CMIP/MOHC/HadGEM3-GC31-MM/piControl/r1i1p1f1/fx/', Delimiter='/', MaxKeys=10)['CommonPrefixes']:
    print("\t", item['Prefix'])

print('Ocean Fixed field locations:')

for item in s3.list_objects_v2(Bucket=bucket, Prefix='CMIP6/CMIP/MOHC/HadGEM3-GC31-MM/piControl/r1i1p1f1/Ofx/', Delimiter='/', MaxKeys=10)['CommonPrefixes']:
    print("\t", item['Prefix'])
    


Hindcast data is organised using the directory structure described above

In [ ]:

def recursive_list_bucket(bucket, prefix, results, print_filenames=False):
    """Return the contents of a bucket descending recursively through the directory tree
    """
    items = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, Delimiter='/')
    try:
        for item in items['CommonPrefixes']:
            results.append(item['Prefix'])
            recursive_list_bucket(bucket, item['Prefix'], results=results)
    except KeyError:  # Object is not a "directory, so mu
        if print_filenames:
            for file in items['Contents']:
                results.append(file['Key'])

results = []
recursive_list_bucket(bucket, 'CMIP6/DCPP/MOHC/HadGEM3-GC31-MM/dcppA-hindcast/s1960-r1i1p1f2/', results)

# Just print the first 15 as an example
print("\n".join(results[:15]))

<!-- DATA PROVIDER INSTRUCTIONS
This section is meant to orient users of your dataset to the formats present in your dataset, particularly if your dataset includes formats that may be unfamiliar to a general data scientist audience. This section should include:

1. Explanation of data format(s) (very common formats can be very briefly described, while less common
   or domain specific formats should include more explanation as well as links to official documentation)
2. Explanation of why the data format was chosen for your dataset
3. Recommendations around software and tooling to work with this data format
4. Explanation of any dataset-specific aspects to your usage of the format
5. Description of AWS services that may be useful to users working with your data
DATA PROVIDER INSTRUCTIONS -->

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

Each variable for each ensemble member consists of a set of netCDF4 files, one for each year. Each has a data variable, coordinate information (time, latitude, longitude, etc.) and a number of global attributes describing how this data fits within the CMIP6 framework.

Tools such as xarray can load multiple files into a single object and support analysis thereafter.


<!-- DATA PROVIDER INSTRUCTIONS
The goal of this section is to demonstrate loading a portion of data from your dataset, and reveal something about its structure.
1. Load an object from S3
2. Show the structure of data in the object
DATA PROVIDER INSTRUCTIONS -->

### Q: Can you show us an example of downloading and loading data from your dataset?

Here we run some simple analysis to get global average temperature from the monthly mean variable "tas" and plot it for 5 ensemble members from 11 different start years for the hindcasts (every 5 years from 1960).

First set up the connection to the bucket through s3fs and write functions to retrieve data and grid cell areas for global averagyin

In [ ]:
# object to interact with the ASDI s3 bucket
fs = s3fs.S3FileSystem()
BUCKET = 'met-office-cmip6'


def load_areacella():
    """
    Load areacella data from the piControl datasets
    """
    url_with_version = fs.ls(f"{BUCKET}/CMIP6/CMIP/MOHC/HadGEM3-GC31-MM/piControl/r1i1p1f1/fx/areacella/gn")
    filenames = fs.ls(url_with_version[0])
    f = [fs.open(file) for file in filenames]
    return xr.open_mfdataset(f)


def load_data(start_year, variant_label, variable="Amon/tas/gn"):
    """
    load data specified by variable (defaults to monthly atmosphere surface temperature)
    """
    url_with_version = fs.ls(f"{BUCKET}/CMIP6/DCPP/MOHC/HadGEM3-GC31-MM/dcppA-hindcast/s{start_year}-{variant_label}/{variable}")
    filenames = fs.ls(url_with_version[0])
    f = [fs.open(file) for file in filenames]
    return xr.open_mfdataset(f)


Load monthly mean data from the first five ensemble members of the hindcasts for every fifth year and compute and store the global average temperature for that month in the dictionary `tas_gm`

In [ ]:
xr.set_options(use_new_combine_kwarg_defaults=True)

areacella = load_areacella()

tas_gm = {}  # global means of tas

ensemble_members = [f'r{realization}i1p1f2' for realization in range(1,6)]
for y in range(1960,2015,5):
    # load data for year and ensemble member
    for ensemble_member in ensemble_members:
        tas = load_data(y, ensemble_member)
        tas_gm[f"{y}-{ensemble_member}"] = tas['tas'].weighted(areacella['areacella']).mean(('lat', 'lon'))
         

Plot the time series with a different colour for each start date.

In [ ]:
colours = ['red', 'orange', 'yellow', 'green', 'blue', 'violet']

for y in range(1960,2015,5):
    # load data for year and ensemble member
    for i, ensemble_member in enumerate(ensemble_members): 
        key = f"{y}-{ensemble_member}"
        if i == 0:
            tas_gm[key].plot(color= colours[y % len(colours)], label=f'{y}', linewidth=0.2)
        else:
            tas_gm[key].plot(color= colours[y % len(colours)], linewidth=0.2)
legend = plt.legend(loc='best')

Use an annual cycle (from the first ensemble member of the 1960 data) to act as a reference and subtract it from all data to see long term trends.

In [ ]:
colours = ['red', 'orange', 'yellow', 'green', 'blue', 'violet']

# Choose to use the annual cycle from the first ensemble member as a reference
reference = tas_gm["1960-r1i1p1f2"].groupby("time.month").mean("time")
for y in range(1960,2015,5):
    # load data for year and ensemble member
    for i, ensemble_member in enumerate(ensemble_members): 
        key = f"{y}-{ensemble_member}"
        ds = tas_gm[key]
        if i == 0:
            (ds.groupby("time.month") - reference).plot(color= colours[y % len(colours)], label=f'{y}', linewidth=0.2)
        else:
            (ds.groupby("time.month") - reference).plot(color= colours[y % len(colours)], linewidth=0.2)
legend = plt.legend(loc='best')
title = plt.title('Near-surface Air Temperature relative to annual cycle of 1960 (r1i1p1f2)')
